## Question 2 

Implement two data descriptors:

1) class NonEmptyStr: only allows non-empty strings (after stripping).
2) class PositiveFloat: only allows floats > 0 (ints are allowed but stored as float).

Then implement a class Product that uses them:
- name = NonEmptyStr()
- price = PositiveFloat()

Requirements:
- Implement __set_name__, __get__, __set__ in each descriptor.
- Store values in the instance dict under a private key derived from the attribute name,
  e.g., f"_{owner.__name__}__{name}" or similar.
- Raise ValueError on invalid assignments.
- Product's attributes should be readable and writable with validation enforced.

No I/O; no external libraries.

In [ ]:
class NonEmptyStr:
    def __set_name__(self, owner, name):
        self.private_name = f"_{owner.__name__}__{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self 
        return instance.__dict__.get(self.private_name)
    
    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise ValueError
        
        stripped_value = value.strip()
        if not stripped_value:
            raise ValueError
        
        instance.__dict__[self.private_name] = stripped_value


class PositiveFloat:
    def __set_name__(self, owner, name): 
        self.private_name = f"_{owner.__name__}__{name}"


    def __get__(self, instance, owner):
        if instance is None:
            return self 
        return getattr(instance, self.private_name, None)
    
    def __set__(self, instance, value):
        if not isinstance(value, (int, float)) or value <= 0:
            raise ValueError
        setattr(instance, self.private_name, float(value))
    


class Product:
    name = NonEmptyStr()
    price = PositiveFloat()

## Question 3 

Implement a decorator factory 
```
retry(retries=3, delay=0.0, backoff=1.0, exceptions=(Exception,))
```
that retries the wrapped function when it raises any of `exceptions`.

Requirements:
- Use `functools.wraps` to preserve metadata.
- On each failure, try again up to `retries` total attempts.
- Sleep between attempts: delay * (backoff ** (attempt_index-1)).
  (The first retry waits `delay`, second waits `delay*backoff`, etc.)
- Expose `wrapper.retries_used` (int) after each call indicating how many attempts were used.
- If all attempts fail, re-raise the last exception.

Notes:
- Tests will pass `delay=0.0` to avoid waiting.
- Do not perform I/O; no logging to files.

In [ ]:
import time
import functools

def retry(retries=3, delay=0.0, backoff=1.0, exceptions=(Exception,)):
    """
    Return a decorator that retries the wrapped function on exceptions.
    Expose wrapper.retries_used after each call.
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            wrapper.retries_used = 0 
            last_exception = None 
            for attempt in range(1, retries + 1):
                wrapper.retries_used = attempt

                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    last_exception = exc 
                    if attempt == retries:
                        raise 
                    sleep_time = delay * (backoff ** (attempt - 1))
                    if sleep_time > 0:
                        time.sleep(sleep_time)
            raise last_exception
        return wrapper
    return decorator 

Within above code, if we do like 

```
@retry(...)
def f(): 


f()
```

Behind the scene, it will run like 

```
decorator = retry(...)
f = decorator(f) = wrapper (as decorator return)

```

So in this case, `wrapper.retries_used` will turn into `f.retries_used`

## Question 4 

Write a decorator `timeit` that measures function runtime.

Requirements:
- Usage: 
    @timeit
    def f(...): ...
- It must call the original function and return its normal result.
- It must store the elapsed time (in seconds, float) on the wrapper as attribute `last_elapsed`.
- Use `functools.wraps` to preserve the original function's metadata (`__name__`, `__doc__`, ...).

Example:
```
>>> @timeit
... def add(a, b): return a + b
>>> r = add(2, 3)
>>> r
5
>>> isinstance(add.last_elapsed, float)
True
>>> add.__name__
'add'
```


In [3]:
import time
import functools

def timeit(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs): 
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        wrapper.last_elapsed = start_time - end_time 
        return result 
    return wrapper 

@timeit 
def add(a,b):
    return a + b

r = add(2,3)
bool_value = isinstance(add.last_elapsed, float)

print(r)
print(bool_value)
print(add.__name__)

5
True
add


## Question 5 

A dataset is available in the folder `animals/` at the root directory.  
This folder contains five class subdirectories: dog, chicken, cat, goose, horse,  
and one file `meta.txt` (ignore this file).

Task:
- Implement a function `build_dataframe(base_path)` that returns a Pandas DataFrame with 2 columns:
    * filename : the image file name (string, not including the directory path)
    * class    : the class name (the subfolder name)
- Requirements:
    * Iterate over all immediate subdirectories of `base_path` (ignore files like meta.txt).
    * Only include files with common image extensions: jpg, jpeg, png, bmp, gif, webp (case-insensitive).
    * Ignore subdirectories inside class folders.
    * The DataFrame does not need to be sorted.
- Example:
    >>> df = build_dataframe("animals")
    >>> set(df.columns) == {"filename", "class"}
    True



In [ ]:
import os
import pandas as pd

def build_dataframe(base_path):
    """
    Return a Pandas DataFrame with 2 columns: filename, class.
    Each row corresponds to an image file under the class subdirectories.
    """
    # TODO
    pass

## Question 6 

A dataset is available in the folder `animals/` at the root directory.  
This folder contains five class subdirectories: dog, chicken, cat, goose, horse,  
and one file `meta.txt` (ignore this file).

Task:
- Implement a function `build_dataframe_with_rgb(base_path)` that returns a Pandas DataFrame with 5 columns:
    * filename : the image file name (string, not including directory path)
    * class    : the class name (the subfolder name)
    * r_mean   : mean value of the red channel (0–255, float or int)
    * g_mean   : mean value of the green channel
    * b_mean   : mean value of the blue channel
- Requirements:
    * Iterate over all immediate subdirectories of `base_path` (ignore files like meta.txt).
    * Only include files with common image extensions: jpg, jpeg, png, bmp, gif, webp (case-insensitive).
    * Ignore subdirectories inside class folders.
    * Use OpenCV (`cv2.imread`) to read images. Remember: cv2 loads images as BGR, so convert to RGB.
- Example:
    >>> df = build_dataframe_with_rgb("animals")
    >>> set(df.columns) == {"filename", "class", "r_mean", "g_mean", "b_mean"}
    True

In [ ]:
import os
import cv2
import pandas as pd

def build_dataframe_with_rgb(base_path):
    """
    Return a Pandas DataFrame with columns:
    filename, class, r_mean, g_mean, b_mean.
    Each row corresponds to one image in the dataset.
    """
    # TODO
    pass

## Question 7 

Implement a function `make_fibo()` that returns a closure `fiboKhông` computing the n-th Fibonacci number.

Requirements:
- Use a closure with an internal DICTIONARY cache (memoization). No globals, no classes, no @lru_cache.
- The cache must persist across multiple calls to the same closure.
- Different closures from `make_fibo()` must have independent caches.

Definitions:
fibo(0) = 0, fibo(1) = 1, fiboKhông = fibo(n-1) + fibo(n-2) for n >= 2.

In [ ]:
def make_fibo():
    """
    Return a closure fibo(n) that computes Fibonacci numbers with memoization.
    Must use an internal DICTIONARY cache inside the closure.
    No globals, no classes, no @lru_cache.
    """
    cache = {}
    def fibo(n):
        if n <= 1: 
            return cache[n]
        cache[n] = fibo(n-1) + fibo(n-2)
        return cache[n]
    return fibo

## Question 8 

Implement a function `make_counter(start=0, step=1)` that returns a closure `counter()`.

Requirements:
- The returned `counter()` keeps internal state via a closure (no global/static/class).
- Each call to `counter()` increases the internal value by `step` and returns the current value.
- Different counters created by `make_counter` must have independent states.

Examples:
```
>>> c = make_counter()        # start=0, step=1
>>> c()
1
>>> c()
2
>>> d = make_counter(10, 5)   # independent
>>> d()
15
>>> c()
3
```

In [10]:
def make_counter(start=0, step=1): 
    """ Return a closure counter() that increments an internal value by step and returns the current value on each call. Must use a closure (no global/static/class state). 
    """ 
    state = start
    def counter():
        nonlocal state
        state += step
        return state
    return counter


c = make_counter()
c()

1

In [11]:
c()

2

In [14]:
d = make_counter(10,5)

In [15]:
d()

15

In [16]:
c()

4

## Question 9 

Implement a function `make_multiplier(factor)` that returns a closure `multiply(x)`.

Requirements:
- The closure must capture the variable `factor`.
- Each call `multiply(x)` returns `x * factor`.
- Different closures must have independent factors.

Example:
```
>>> times3 = make_multiplier(3)
>>> times3(4)
12
>>> times5 = make_multiplier(5)
>>> times5(2)
10
```

In [17]:
def make_multiplier(factor):
    """
    Return a closure multiply(x) that returns x * factor.
    Must capture factor via closure.
    """
    cache_factor = factor 
    def multiply(x):
        return x * cache_factor
    return multiply

times3 = make_multiplier(3)
times3(4)
    

12

In [18]:
times5 = make_multiplier(5)
times5(2)

10

## Question 10 

Write a function `list_files(path)` that returns a sorted list of file names 
(including extension) in the given directory. 

- Do not include subdirectories in the result. 
- You may use modules `os` or `pathlib`. 

In [19]:
import os
from pathlib import Path

def list_files(path):
    """
    Return a sorted list of file names in the given directory.
    Ignore subdirectories.
    """
    path = Path(path)
    files = [p.name for p in path.iterdir() if p.is_file()]

    return sorted(files)

list_files(".")

['.gitignore',
 'code-mix.ipynb',
 'index_slicing.ipynb',
 'numpy-practice.ipynb',
 'quiz-1.ipynb',
 'quiz-2.ipynb',
 'quiz-3.ipynb',
 'requirements.txt']

## Question 11 

Write a decorator factory `repeatKhông` that repeats the execution of a function `n` times.

Requirements:
- Usage:
```
    @repeat(3)
    def hello(): print("hi")
```
- If the function returns a value, only the last result is returned.
- Use functools.wraps to preserve metadata.



In [27]:
import functools

def repeat(n):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def hello():
    print("Hi")

hello()
    

Hi
Hi
Hi


## Question 12 

Write a decorator `uppercase` that converts the return value of a function into uppercase if it is a string.

Requirements:
- Usage: 
    @uppercase
    def greet(name): return "Hello " + name
- If the result is not a string, return it unchanged.
- Use functools.wraps to preserve metadata.

In [31]:
import functools

def uppercase(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        if not isinstance(result, str):
            return result 
        else:
            return result.upper()
    return wrapper 

@uppercase 
def greet(name): 
    return "Hello " + name 

greet("hung")

'HELLO HUNG'

## Question 13

A dataset is available in the folder `animals/` at the root directory.  
This folder contains five class subdirectories: dog, chicken, cat, goose, horse,  
and one file `meta.txt` (ignore this file).

Task:
- Implement a function `build_dataframe_with_size(base_path)` that returns a Pandas DataFrame with 4 columns:
    * filename : the image file name (string, not including directory path)
    * class    : the class name (the subfolder name)
    * width    : image width in pixels (integer)
    * height   : image height in pixels (integer)
- Requirements:
    * Iterate over all immediate subdirectories of `base_path` (ignore files like meta.txt).
    * Only include files with common image extensions: jpg, jpeg, png, bmp, gif, webp (case-insensitive).
    * Ignore subdirectories inside class folders.
    * Use Pillow (`from PIL import Image`) to read image size.
- Example:
    >>> df = build_dataframe_with_size("animals")
    >>> set(df.columns) == {"filename", "class", "width", "height"}
    True



In [ ]:
import os
import pandas as pd
from PIL import Image

def build_dataframe_with_size(base_path):
    """
    Return a DataFrame with columns: filename, class, width, height.
    Each row corresponds to one image in the dataset.
    """
    # TODO
    pass

## Question 14 

A dataset is available in the folder `animals/` at the root directory.  
This folder contains five class subdirectories: dog, chicken, cat, goose, horse,  
and one file `meta.txt` (ignore this file).

Task:
- Implement a function `count_images_per_class(base_path)` that returns a dictionary mapping class_name -> number_of_images.
- Requirements:
  * Iterate over all immediate subdirectories of `base_path` (ignore files like meta.txt).
  * Count only image files inside each class folder. Ignore subdirectories.
  * Consider common image extensions: jpg, jpeg, png, bmp, gif, webp (case-insensitive).
  * Return a dictionary {class_name: count} with integer counts.

Example:
>>> res = count_images_per_class("animals")
>>> isinstance(res, dict)
True

In [ ]:
import os

def count_images_per_class(base_path):
    """
    Return a dict {class_name: number_of_images} by scanning the immediate
    subdirectories of `base_path` and counting image files inside each class folder.

    Image extensions considered: jpg, jpeg, png, bmp, gif, webp (case-insensitive).
    Ignore files at base_path (e.g., meta.txt) and ignore subdirectories inside classes.
    """
    # TODO: implement
    pass

## Question 15 

A dataset is available in the folder `animals/` at the root directory.
This folder contains five class subdirectories: dog, chicken, cat, goose, horse,
and one file `meta.txt` (ignore this file).

Task:
- Implement a function `build_rgb_features_and_labels(base_path)` that returns a tuple `(X, y, classes)`:
    * X       : a list of feature vectors [[r_mean, g_mean, b_mean], ...] computed from each image
    * y       : a list of integer labels corresponding to each row in X
    * classes : a list of class names, where `y` uses index in this list (i.e., label encoding)
- Requirements:
    * Use OpenCV (`cv2.imread`) to read images. Remember: cv2 loads images as BGR, so convert to RGB.
    * Consider only files with extensions: jpg, jpeg, png, bmp, gif, webp (case-insensitive).
    * Iterate only immediate subdirectories of `base_path`; ignore files like meta.txt.
    * Ensure **deterministic label encoding** by sorting class names alphabetically to build `classes`.
    * The order of samples in X and y does not matter, but y[i] must match X[i] and classes.

Example:
>>> X, y, classes = build_rgb_features_and_labels("animals")
>>> isinstance(X, list) and isinstance(y, list) and isinstance(classes, list)
True

In [ ]:
import os
import cv2

def build_rgb_features_and_labels(base_path):
    """
    Return (X, y, classes):
      - X: list of [r_mean, g_mean, b_mean] for each image
      - y: list of integer labels aligned with X
      - classes: sorted list of class names; labels are indices in this list
    Use cv2 to read images (BGR) and convert to RGB before computing means.
    """
    # TODO
    pass